# Pairwise evaluation with Kaggle Models

This notebook performs the assignment's required model-based evaluation through `kaggle_benchmarks`. It judges the saved pairwise files only; it does not load training checkpoints or regenerate responses.


## Kaggle prerequisite

Create this notebook as a **Kaggle utility benchmark** and enable Kaggle model access. The default judge is `google/gemini-3.6-flash`, as suggested by the assignment manual. If Kaggle does not expose that model, replace `JUDGE_MODEL` below with another Kaggle-supported model-proxy judge and record the exact model in the report.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/syeddaniyalg/cs5326-pa2"
REPO_ROOT = Path("/kaggle/working/cs5326-pa2")

if REPO_ROOT.exists():
    print(f"Using existing repository: {REPO_ROOT}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
print(f"Repository: {REPO_ROOT}")


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT), "--no-deps"],
    check=True,
)


In [ ]:
import importlib
import json
import os
import shlex

SCRIPTS = REPO_ROOT / "scripts"
OUTPUTS = REPO_ROOT / "outputs"
PROCESS_ENV = os.environ.copy()
PROCESS_ENV["PYTHONPATH"] = str(REPO_ROOT)

def run_script(script_name, *arguments):
    command = [sys.executable, str(SCRIPTS / script_name), *map(str, arguments)]
    print("Running:", shlex.join(command), flush=True)
    subprocess.run(command, cwd=REPO_ROOT, env=PROCESS_ENV, check=True)

os.chdir(REPO_ROOT)
kaggle_benchmarks = importlib.import_module("kaggle_benchmarks")
print("kaggle_benchmarks: available")


In [ ]:
required_pairs = {
    OUTPUTS / "eval" / "dpo_uf_vs_sft_pairs.jsonl": 300,
    OUTPUTS / "eval" / "dpo_hh_vs_sft_pairs.jsonl": 300,
    OUTPUTS / "eval" / "lora_vs_base_pairs.jsonl": 40,
}

for path, expected_rows in required_pairs.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Commit the pair files or attach/copy them into outputs/eval/."
        )
    rows = [line for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if len(rows) != expected_rows:
        raise ValueError(f"{path.name}: expected {expected_rows} rows, found {len(rows)}")
    print(f"{path.name}: {len(rows)} pairs")


In [ ]:
JUDGE_MODEL = "google/gemini-3.6-flash"
JUDGE_COMMON = [
    "--judge_model", JUDGE_MODEL,
    "--batch_size", "5",
    "--sleep_seconds", "5",
    "--max_retries", "3",
    "--max_chars", "2500",
    "--reuse_judgments",
]
print("Kaggle judge:", JUDGE_MODEL)


## Task 7: resumable Kaggle pairwise judging

Each command checkpoints judgments as they are produced. Rerunning a cell reuses completed pair IDs and continues from pending or unresolved items.


In [ ]:
run_script(
    "judge_pairwise_kaggle.py",
    "--pairs_path", "outputs/eval/dpo_uf_vs_sft_pairs.jsonl",
    "--output_path", "outputs/eval/dpo_uf_vs_sft_judged.jsonl",
    "--rubric", "general",
    *JUDGE_COMMON,
)


In [ ]:
run_script(
    "judge_pairwise_kaggle.py",
    "--pairs_path", "outputs/eval/dpo_hh_vs_sft_pairs.jsonl",
    "--output_path", "outputs/eval/dpo_hh_vs_sft_judged.jsonl",
    "--rubric", "safety",
    *JUDGE_COMMON,
)


In [ ]:
run_script(
    "judge_pairwise_kaggle.py",
    "--pairs_path", "outputs/eval/lora_vs_base_pairs.jsonl",
    "--output_path", "outputs/eval/lora_vs_base_style_judged.jsonl",
    "--rubric", "persona_style",
    *JUDGE_COMMON,
)


In [ ]:
run_script(
    "judge_pairwise_kaggle.py",
    "--pairs_path", "outputs/eval/lora_vs_base_pairs.jsonl",
    "--output_path", "outputs/eval/lora_vs_base_quality_judged.jsonl",
    "--rubric", "persona_quality",
    *JUDGE_COMMON,
)


## Summaries, figures, and report tables


In [ ]:
run_script("make_figures.py", "--output_root", "outputs")
run_script("make_tables.py", "--output_root", "outputs")


In [ ]:
from IPython.display import Image, Markdown, display

for path in sorted((OUTPUTS / "figures").glob("eval_*.png")):
    display(Image(filename=str(path)))

for path in sorted((OUTPUTS / "tables").glob("*.md")):
    display(Markdown(path.read_text(encoding="utf-8")))


## Save evaluation artifacts

The archive is written directly under `/kaggle/working` for download or publication as a Kaggle output dataset.


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

archive_path = Path("/kaggle/working/evaluation_outputs.zip")
roots = [OUTPUTS / "eval", OUTPUTS / "figures", OUTPUTS / "tables"]
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED, compresslevel=6) as archive:
    for root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(REPO_ROOT).as_posix())
print(f"Saved {archive_path} ({archive_path.stat().st_size / 2**20:.1f} MiB)")
